# Huấn luyện ViT5 warm-start – Phase 1 trên Google Colab

Notebook này chạy `configs/vit5_warmstart_phase_1.yaml` với GPU Colab và lưu checkpoint, log, metrics vào Google Drive. Trước khi chạy, chọn **Runtime → Change runtime type → GPU**.

Dữ liệu được đọc từ `MyDrive/sumarization/data/phase_1` nếu thư mục này đã tồn tại; nếu không notebook dùng dữ liệu trong repository vừa clone.

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

from google.colab import drive

REPO_URL = 'https://github.com/dungcony/sumarization.git'
REPO_DIR = Path('/content/sumarization')
DRIVE_ROOT = Path('/content/drive/MyDrive/sumarization')
HF_CACHE_DIR = DRIVE_ROOT / 'hf_cache'

drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Giữ model/tokenizer đã tải trên Drive để không download lại ở runtime sau.
os.environ['HF_HOME'] = str(HF_CACHE_DIR)

def run(command: list[str]) -> None:
    print('$', ' '.join(command))
    subprocess.run(command, check=True)

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').exists():
        raise RuntimeError(f'{REPO_DIR} đã tồn tại nhưng không phải Git repository.')
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print(f'Đang làm việc tại: {Path.cwd()}')
print(f'Checkpoint và logs sẽ lưu tại: {DRIVE_ROOT}')
print(f'Hugging Face cache: {HF_CACHE_DIR}')

In [ ]:
%pip install -q --upgrade pip
%pip install -q -e .

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'Không tìm thấy GPU. Hãy chọn Runtime → Change runtime type → GPU, rồi chạy lại từ đầu.'
    )

GPU_NAME = torch.cuda.get_device_name(0)
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
print(f'GPU: {GPU_NAME}')
print(f'Precision: {PRECISION}')
print(f'CUDA: {torch.version.cuda}')

## Cấu hình dữ liệu và đầu ra

Để không phải clone dữ liệu ở các lần chạy sau, hãy upload/copy thư mục `data/phase_1` vào `MyDrive/sumarization/data/phase_1`. Nếu chưa có, notebook tự dùng `data/phase_1` trong repository.

Mặc định dùng gradient checkpointing và batch size nhỏ hơn để chạy ổn định trên T4 16 GB, nhưng giữ effective batch size là 8 như YAML gốc. Với A100/L4 còn đủ bộ nhớ, có thể đổi `TRAIN_BATCH_SIZE = 4`, `EVAL_BATCH_SIZE = 8`, `GRADIENT_ACCUMULATION_STEPS = 2`.

In [ ]:
from glob import glob

from src.config import apply_overrides, load_config
from transformers.trainer_utils import get_last_checkpoint

CONFIG_FILE = REPO_DIR / 'configs/vit5_warmstart_phase_1.yaml'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data'
REPO_DATA_ROOT = REPO_DIR / 'data'
DATA_ROOT = DRIVE_DATA_ROOT if (DRIVE_DATA_ROOT / 'phase_1').exists() else REPO_DATA_ROOT
OUTPUT_DIR = DRIVE_ROOT / 'outputs_phase_1/vit5_warmstart'

# Thiết lập an toàn cho GPU T4 16 GB; effective train batch = 2 × 4 = 8.
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
AUTO_RESUME = True  # Đổi thành False và dùng OUTPUT_DIR mới nếu muốn train lại từ đầu.

# Checkpoint được Trainer lưu mỗi 200 bước vào OUTPUT_DIR (Google Drive).
# Sau khi Colab ngắt, chạy lại notebook: checkpoint mới nhất sẽ tự được dùng.
RESUME_FROM_CHECKPOINT = (
    get_last_checkpoint(str(OUTPUT_DIR))
    if AUTO_RESUME and OUTPUT_DIR.exists()
    else None
)

data_patterns = {
    'train': str(DATA_ROOT / 'phase_1/train_*.parquet'),
    'validation': str(DATA_ROOT / 'phase_1/validation_*.parquet'),
    'test': str(DATA_ROOT / 'phase_1/test_*.parquet'),
}
for split, pattern in data_patterns.items():
    files = glob(pattern)
    if not files:
        raise FileNotFoundError(f'Không tìm thấy dữ liệu {split}: {pattern}')
    print(f'{split}: {len(files)} file(s)')

config = load_config(CONFIG_FILE)
config = apply_overrides(config, {
    'data.train_file': data_patterns['train'],
    'data.valid_file': data_patterns['validation'],
    'data.test_file': data_patterns['test'],
    'model.cache_dir': str(HF_CACHE_DIR),
    'training.output_dir': str(OUTPUT_DIR),
    'training.precision': PRECISION,
    'training.optim': 'adamw_torch',
    'training.gradient_checkpointing': True,
    'training.per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'training.per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'training.gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'training.resume_from_checkpoint': RESUME_FROM_CHECKPOINT,
})

print(f'Config: {CONFIG_FILE.name}')
print(f'Model: {config.model.name_or_path}')
print(f'Data root: {DATA_ROOT}')
print(f'Output: {config.training.output_dir}')
print(f'Effective batch size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}')
if RESUME_FROM_CHECKPOINT:
    print(f'↩️  Sẽ tiếp tục từ: {RESUME_FROM_CHECKPOINT}')
else:
    print('🆕 Không có checkpoint cũ; sẽ train từ đầu.')

In [ ]:
from src.data import load_and_preprocess
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import count_parameters, format_number, set_seed

set_seed(config.training.seed)
tokenizer = load_tokenizer(config.model)
model = load_model(config.model, tokenizer, config.generation)

if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)
if config.training.freeze_encoder:
    freeze_encoder(model)
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(
    f"Model: {format_number(params['total'])} tham số; "
    f"{format_number(params['trainable'])} có thể train"
)

datasets = load_and_preprocess(tokenizer, config.data)
for split, dataset in datasets.items():
    print(f'{split}: {len(dataset):,} mẫu')

In [ ]:
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.evaluator import build_compute_metrics

tc = config.training
Path(tc.output_dir).mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,
    label_smoothing_factor=tc.label_smoothing_factor,
    fp16=(tc.precision == 'fp16'),
    bf16=(tc.precision == 'bf16'),
    gradient_checkpointing=tc.gradient_checkpointing,
    eval_strategy=tc.eval_strategy,
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,
    logging_steps=tc.logging_steps,
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,
    predict_with_generate=True,
    generation_max_length=config.generation.max_length,
    generation_num_beams=config.generation.num_beams,
    report_to=['tensorboard'],
    logging_dir=str(Path(tc.output_dir) / 'logs'),
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)
callbacks = []
if tc.early_stopping_patience > 0:
    callbacks.append(EarlyStoppingCallback(tc.early_stopping_patience))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(tokenizer),
    callbacks=callbacks,
)

print('Trainer đã sẵn sàng.')

## Huấn luyện

Nếu Colab bị ngắt, hãy chạy lại các cell từ đầu đến đây. Notebook tự tìm checkpoint mới nhất trong Google Drive và tiếp tục từ đó. Để cố ý train lại từ đầu, đổi `AUTO_RESUME = False` và dùng `OUTPUT_DIR` mới trong cell cấu hình.

In [ ]:
import time

from src.utils import format_duration

print('=' * 60)
print(f'BẮT ĐẦU TRAIN {config.phase.name}')
print(f'Epochs: {tc.num_train_epochs} | LR: {tc.learning_rate} | Precision: {tc.precision}')
print('=' * 60)

start_time = time.time()
train_result = trainer.train(resume_from_checkpoint=tc.resume_from_checkpoint)
print(f'Hoàn thành sau {format_duration(time.time() - start_time)}')

In [ ]:
from src.config import config_to_dict
from src.utils import save_json

output_dir = Path(tc.output_dir)
best_dir = output_dir / 'best'
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))

eval_results = trainer.evaluate(metric_key_prefix='eval')
test_results = {}
if 'test' in datasets:
    test_results = trainer.evaluate(
        eval_dataset=datasets['test'],
        metric_key_prefix='test',
    )

train_metrics = train_result.metrics
train_metrics['train_runtime_formatted'] = format_duration(
    train_metrics.get('train_runtime', 0)
)
save_json(train_metrics, output_dir / 'train_results.json')
save_json(eval_results, output_dir / 'eval_results.json')
if test_results:
    save_json(test_results, output_dir / 'test_results.json')
save_json(config_to_dict(config), output_dir / 'resolved_config.json')

for title, results in [('Validation', eval_results), ('Test', test_results)]:
    if results:
        print(f'\n{title}')
        for key, value in sorted(results.items()):
            if isinstance(value, float):
                print(f'  {key}: {value:.4f}')

print(f'✅ Model tốt nhất: {best_dir}')

In [ ]:
from src.predict import summarize

test_text = (
    'Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng trí tuệ '
    'nhân tạo tại Việt Nam giai đoạn 2025-2030. Đề án tập trung vào y tế, '
    'giáo dục, nông nghiệp, giao thông và sản xuất công nghiệp.'
)
summary = summarize(text=test_text, model_path=str(best_dir), config=config)
print('Bài gốc:', test_text)
print('\nTóm tắt:', summary)